# Generate Train / Val / Test Splits

Run this **once** after `generate_trajectories.ipynb` finishes. The outputs
(`splits.parquet`, `feature_stats.csv`, and the rebalanced trajectories
parquet) are shared with the group — everyone trains on the same data so
results are comparable.

**Vessel-level split**: each MMSI goes entirely into one split.
This tests whether the model generalizes to **unseen ships**, not just
unseen timesteps of known ships.

**Train-set rebalancing**: the raw AIS data is dominated by moored / drifting
vessels (~85% of windows have mean SOG < 0.5 kt on West Coast). To keep the
model from collapsing to "predict no motion", we drop trips whose mean SOG
is below `MIN_TRAIN_TRIP_SOG` — but **only inside the training split**.
Val and test trips are kept untouched so reported metrics still reflect the
real distribution.

**Train-only stats**: mean/std/min/max are computed on the (rebalanced)
training set only. Using val/test would leak their statistics into the
model's input normalization.

## 1. Configuration

`INTERP_FILE` must match what you produced in `generate_trajectories.ipynb`.

In [ ]:
import numpy as np
import pandas as pd

OUT_DIR = "processed"

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  >>> Must match the file generated by generate_trajectories.ipynb <<<║
# ╚══════════════════════════════════════════════════════════════════════╝
TIMESTEP_MIN = 3
REGION = "West Coast"
INTERP_FILE  = f"{OUT_DIR}/ais_trajectories_{REGION}_{TIMESTEP_MIN}min.parquet"

SPLITS_FILE = f"{OUT_DIR}/splits_{REGION}_{TIMESTEP_MIN}min.parquet"
STATS_FILE  = f"{OUT_DIR}/feature_stats_{REGION}_{TIMESTEP_MIN}min.csv"

# Split ratios — remaining 15% becomes test
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

# DO NOT CHANGE — everyone must get the same random assignment
SEED = 42

# Train-set rebalancing — drop training trips whose mean SOG (knots) is below
# this threshold. Set to 0 (or BALANCE_TRAIN=False) to keep all trips.
BALANCE_TRAIN = True
MIN_TRAIN_TRIP_SOG = 0.5

# What fraction of slow training trips to KEEP (instead of drop). Default 0.0
# = drop them all (the original aggressive rebalance). Setting this to e.g.
# 0.1 keeps 10% of slow trips so the model still sees some always-slow
# vessels (drifting fishing boats, anchored ships swinging on chain) instead
# of only "moored sections inside otherwise-moving trips".
#
# NOTE: not idempotent when > 0 — re-running this notebook will subsample
# the *already-subsampled* slow set, shrinking it further. To change this
# fraction, re-run `merge_monthly.ipynb` first to restore the unfiltered
# trajectories parquet, then re-run this notebook.
KEEP_SLOW_FRACTION = 0.0

## 2. Load interpolated trajectories

In [ ]:
df = pd.read_parquet(INTERP_FILE)
print(f"Loaded: {len(df):,} rows")
print(f"  Vessels: {df['mmsi'].nunique():,}")
print(f"  Trips:   {df['trip_id'].nunique():,}")
print(f"  Date range: {df['base_date_time'].min()} → {df['base_date_time'].max()}")

## 3. Vessel-level split

Shuffle MMSIs with a fixed seed, then slice into train / val / test.
The sort before shuffling makes the result deterministic across machines.

In [ ]:
rng = np.random.default_rng(SEED)
mmsis = np.array(sorted(df["mmsi"].unique()))   # sort for reproducibility
rng.shuffle(mmsis)

n = len(mmsis)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)

labels = np.empty(n, dtype=object)
labels[:n_train]                = "train"
labels[n_train : n_train + n_val] = "val"
labels[n_train + n_val :]       = "test"

splits = pd.DataFrame({"mmsi": mmsis.astype("int64"), "split": labels})
splits.to_parquet(SPLITS_FILE, index=False)

print(f"Split summary (by vessel):")
print(splits["split"].value_counts().to_string())
print(f"\nSaved → {SPLITS_FILE}")

## 4. Rebalance training trips (drop slow ones)

Filter out trips whose mean SOG is below `MIN_TRAIN_TRIP_SOG` knots — but
only for vessels assigned to **train**. Val and test are left untouched.
We then overwrite `INTERP_FILE` with the filtered table so downstream code
(the dataloader) doesn't need to know about this step.

**Re-running this notebook is idempotent**: after one pass, the slow training
trips are gone, so a second pass drops nothing. To restore the unfiltered
parquet, re-run `generate_trajectories.ipynb`.

In [ ]:
if BALANCE_TRAIN:
    train_mmsi = splits.loc[splits["split"] == "train", "mmsi"]
    is_train = df["mmsi"].isin(train_mmsi)

    # Mean SOG per trip on the train side only. Trips whose SOG is entirely
    # NaN (rare, but interpolation can't help if the source had no SOG) are
    # treated as moored and dropped — `fillna(0)` makes them fall below the
    # threshold.
    train_trip_sog = df.loc[is_train].groupby("trip_id")["sog"].mean()
    slow_trips = train_trip_sog[
        train_trip_sog.fillna(0) < MIN_TRAIN_TRIP_SOG
    ].index

    # Keep a fraction of slow trips (so the model is exposed to always-slow
    # vessel dynamics). Sampling is deterministic with SEED.
    if KEEP_SLOW_FRACTION > 0 and len(slow_trips):
        rng_slow = np.random.default_rng(SEED)
        n_keep = int(round(len(slow_trips) * KEEP_SLOW_FRACTION))
        kept_back = rng_slow.choice(slow_trips.to_numpy(),
                                    size=n_keep, replace=False)
        slow_trips = slow_trips.difference(pd.Index(kept_back))
        print(f"  KEEP_SLOW_FRACTION={KEEP_SLOW_FRACTION:.2f} → keeping "
              f"{n_keep:,} slow trips back in the train set")

    n_train_trips_before = is_train.groupby(df["trip_id"]).any().sum()  # train trip count
    n_rows_before = len(df)

    keep = ~(is_train & df["trip_id"].isin(slow_trips))
    df = df.loc[keep].reset_index(drop=True)

    print(f"Dropped {len(slow_trips):,} train trips with mean SOG < "
          f"{MIN_TRAIN_TRIP_SOG} kt")
    print(f"  Train trips: {n_train_trips_before:,} → "
          f"{n_train_trips_before - len(slow_trips):,}")
    print(f"  Trajectories: {n_rows_before:,} → {len(df):,} rows "
          f"({n_rows_before - len(df):,} removed)")

    df.to_parquet(INTERP_FILE, index=False)
    print(f"  Overwrote → {INTERP_FILE}")
else:
    print("BALANCE_TRAIN=False — keeping all training trips")

# Row-level counts (what the DataLoader will actually see)
merged = df.merge(splits, on="mmsi", how="left")
print(f"\nRow-level counts after rebalance (DataLoader input):")
print(merged["split"].value_counts().to_string())

## 5. Feature statistics (train-only, post-rebalance)

Saved to `feature_stats.csv`. The dataloader uses these to normalize
model inputs: `x_norm = (x - mean) / std`.

We include `cog` raw statistics for completeness, but the dataloader will
instead split `cog` into `cos(cog)` and `sin(cog)` to handle the 0°/360°
wraparound — those derived features don't need normalization (both already
in [−1, 1]).

In [ ]:
train = merged[merged["split"] == "train"]
# Stats for every numeric feature the dataloader might expose, so teammates
# can pick any subset in dataloader.get_dataloaders(features=...) without
# recomputing stats from their (possibly val/test) data.
feature_cols = ["latitude", "longitude", "sog", "cog", "heading",
                "length", "width", "draft"]
stats = train[feature_cols].describe().T[["mean", "std", "min", "max"]]
stats.to_csv(STATS_FILE)

print(f"Feature stats (computed on {len(train):,} training rows):")
print(stats)
print(f"\nSaved → {STATS_FILE}")

## 6. Verification

Sanity checks that everyone will run to confirm the splits match what the
group agreed on.

In [ ]:
# 1. No vessel appears in two splits
vessel_to_splits = splits.groupby("mmsi")["split"].nunique()
assert vessel_to_splits.max() == 1, "ERROR: vessel appears in multiple splits"
print("✓ No vessel appears in multiple splits")

# 2. All vessels remaining in the (rebalanced) data are assigned a split.
#    Some training MMSIs may have lost every trip during rebalance — that's
#    fine, splits.parquet still lists them, they just contribute 0 rows.
unassigned = set(df["mmsi"].unique()) - set(splits["mmsi"])
assert len(unassigned) == 0, f"ERROR: {len(unassigned)} vessels unassigned"
print("✓ All remaining vessels assigned to a split")

# 3. Fractions are close to target (vessel-level — unaffected by rebalancing)
actual = splits["split"].value_counts(normalize=True)
print(f"\nActual vessel fractions: {actual.to_dict()}")
print("  (targets: train=0.70, val=0.15, test=0.15)")

# 4. After rebalance, the only train trips below the threshold should be the
#    ones we deliberately kept back via KEEP_SLOW_FRACTION.
if BALANCE_TRAIN:
    train_rows = merged[merged["split"] == "train"]
    if len(train_rows):
        post_trip_sog = train_rows.groupby("trip_id")["sog"].mean()
        n_below = int((post_trip_sog.fillna(0) < MIN_TRAIN_TRIP_SOG).sum())
        if KEEP_SLOW_FRACTION == 0.0:
            assert n_below == 0, f"ERROR: {n_below} train trips still below threshold"
            print(f"✓ All {len(post_trip_sog):,} remaining train trips have "
                  f"mean SOG ≥ {MIN_TRAIN_TRIP_SOG} kt")
        else:
            print(f"  {n_below:,} slow train trips kept by design "
                  f"(KEEP_SLOW_FRACTION={KEEP_SLOW_FRACTION:.2f}); "
                  f"{len(post_trip_sog) - n_below:,} fast trips")